# ANKA-50M C0 — Baseline Throughput

> **Status:** Historical baseline benchmark

## Purpose

Define the original C0 decoder-only Transformer, verify its exact 49,960,320-parameter budget, and measure model/training throughput under controlled GPU settings.

## Execution guidance

Select an A100 runtime and execute the sections in order. Keep GPU, batch size, sequence length, precision, and compile settings fixed when comparing measurements.

## Primary outputs

Parameter checks, VRAM use, step time, and tokens per second.

## Reproducibility notes

- Code-cell outputs and execution counters are intentionally cleared in the public release.
- Seeds, persistent paths, and checkpoint-resume behavior remain in the code.
- Review path constants before execution; historical notebooks may reference earlier run directories.
- Execute sections from top to bottom unless a section explicitly describes an independent experiment arm.


## 1. Inspect the GPU environment

Initializes the runtime, deterministic settings, dependencies, and persistent paths required by later sections.


In [ ]:
# CELL 1 — Check environment and A100
import sys, platform, torch

assert torch.cuda.is_available(), 'GPU not found. Select Colab > Runtime > Change runtime type > A100 GPU.'
device = torch.device('cuda')
gpu_name = torch.cuda.get_device_name(0)

# TF32 uses A100 Tensor Cores more efficiently for float32 matrix operations.
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print('Python :', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.version.cuda)
print('GPU    :', gpu_name)
print('BF16   :', torch.cuda.is_bf16_supported())
print('VRAM   :', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GB')

## 2. Define the complete C0 baseline architecture

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# CELL 2 — THE ENTIRE ANKA-50M C0 MODEL IS IN THIS CELL
# This cell only defines the baseline architecture; Reflect, Morph, and WorldPairs are not here.

from dataclasses import dataclass
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class ANKAConfig:
    # 15,360 is currently the candidate vocabulary. If the tokenizer experiment changes, the parameter calculation will be revised.
    vocab_size: int = 15_360
    # 384 hidden, with 6 attention heads, gives 64 dimensions per head.
    d_model: int = 384
    n_layers: int = 28
    n_heads: int = 6
    # 2 KV heads share 6 query heads in groups of three (GQA).
    n_kv_heads: int = 2
    # 1024 for SwiGLU: close to 384 * 8/3 and a hardware-friendly integer.
    d_ff: int = 1024
    max_seq_len: int = 2048
    rope_theta: float = 10_000.0
    norm_eps: float = 1e-5


class RMSNorm(nn.Module):
    # RMSNorm does not use the mean subtraction step in LayerNorm; common in modern decoders.
    def __init__(self, dim: int, eps: float):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        # Performing norm calculation in float32 improves numerical stability under BF16.
        scale = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * scale).to(dtype=x.dtype) * self.weight


def build_rope_cache(max_seq_len: int, head_dim: int, theta: float):
    # RoPE frequencies are not parameters; it adds positional information to the model via rotations.
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    positions = torch.arange(max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    # x shape: [batch, head, sequence, head_dim]. Processes even/odd channels as 2B rotations.
    x_float = x.float()
    x_even, x_odd = x_float[..., 0::2], x_float[..., 1::2]
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    out_even = x_even * cos - x_odd * sin
    out_odd = x_even * sin + x_odd * cos
    return torch.stack((out_even, out_odd), dim=-1).flatten(-2).to(dtype=x.dtype)


class GroupedQueryAttention(nn.Module):
    def __init__(self, cfg: ANKAConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        assert cfg.n_heads % cfg.n_kv_heads == 0
        self.n_heads = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.head_dim = cfg.d_model // cfg.n_heads

        # We don't use bias. Separate Q/K/V projections keep parameter counting clear and testable.
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(cfg.n_heads * self.head_dim, cfg.d_model, bias=False)

        cos, sin = build_rope_cache(cfg.max_seq_len, self.head_dim, cfg.rope_theta)
        # persistent=False: RoPE cache does not inflate checkpoint weight size.
        self.register_buffer('rope_cos', cos, persistent=False)
        self.register_buffer('rope_sin', sin, persistent=False)

    def forward(self, x):
        batch, seq_len, _ = x.shape
        q = self.q_proj(x).view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

        cos = self.rope_cos[:seq_len]
        sin = self.rope_sin[:seq_len]
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # PyTorch SDPA can automatically select Flash Attention kernel on compatible CUDA versions.
        # enable_gqa applies 6Q/2KV GQA without physically copying KV heads.
        y = F.scaled_dot_product_attention(
            q, k, v, dropout_p=0.0, is_causal=True, enable_gqa=True
        )
        y = y.transpose(1, 2).contiguous().view(batch, seq_len, -1)
        return self.o_proj(y)


class SwiGLU(nn.Module):
    def __init__(self, cfg: ANKAConfig):
        super().__init__()
        # gate and up are two separate input projections, down maps back to hidden dimension.
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class ANKABlock(nn.Module):
    def __init__(self, cfg: ANKAConfig):
        super().__init__()
        # Pre-norm residual structure provides safer optimization behavior in deep networks.
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GroupedQueryAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg)

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        x = x + self.ffn(self.ffn_norm(x))
        return x


class ANKA50M(nn.Module):
    def __init__(self, cfg: ANKAConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_embeddings = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList([ANKABlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        self.apply(self._init_weights)
        # Weight tying: the output matrix uses the same parameter as the embedding.
        # Without this sharing, there would be an additional 15,360*384 parameters, exceeding the 50M limit.
        self.lm_head.weight = self.tok_embeddings.weight

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids):
        if input_ids.shape[1] > self.cfg.max_seq_len:
            raise ValueError(f'Sequence length {input_ids.shape[1]} > max_seq_len {self.cfg.max_seq_len}')
        x = self.tok_embeddings(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.final_norm(x)
        # We generate logits at all positions; this is a speed measurement closer to training-prefill cost.
        return self.lm_head(x)

print('ANKA-50M C0 classes defined.')

## 3. Instantiate C0 and verify the parameter limit

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# CELL 3 — Set up the model, convert to BF16, and verify parameter limit
cfg = ANKAConfig()
model = ANKA50M(cfg).to(device=device, dtype=torch.bfloat16).eval()

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
expected_params = 49_960_320

# Does weight tying really use the same tensor storage?
assert model.lm_head.weight.data_ptr() == model.tok_embeddings.weight.data_ptr()
assert trainable_params == expected_params, (trainable_params, expected_params)
assert trainable_params < 50_000_000

print(f'Total parameters      : {all_params:,}')
print(f'Trainable parameters : {trainable_params:,}')
print(f'Remaining budget below 50M : {50_000_000 - trainable_params:,}')
print('Weight tying          : PASS')
print('Parameter limit       : PASS')

## 4. Define the compiled training step

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# CELL 4B — training_step FIX
# Overwrites the previous function with a clean and correctly indented version.

import torch
import torch.nn.functional as F


def training_step(input_ids):
    optimizer.zero_grad(set_to_none=True)

    # Model input can remain int32.
    model_inputs = input_ids[:, :-1]

    # CrossEntropy targets must be int64/long.
    targets = input_ids[:, 1:].long()

    # Forward
    logits = model(model_inputs)

    # Next-token prediction loss
    loss = F.cross_entropy(
        logits.reshape(-1, cfg.vocab_size),
        targets.reshape(-1),
    )

    # Backward
    loss.backward()

    # Optional gradient clipping
    if globals().get("USE_GRAD_CLIP", False):
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            globals().get("MAX_GRAD_NORM", 1.0),
        )

    # Weight update
    optimizer.step()

    return float(loss.detach())


print("training_step fixed: PASS")

## 5. Measure end-to-end training throughput

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:

# SINGLE CELL — FULL TRAINING THROUGHPUT TEST INCLUDING DATALOADER
#
# Single dependency:
#     model
#
# This cell creates all of the following within itself:
# - device and config detection
# - optimizer
# - training_step
# - random token dataset
# - DataLoader
# - warm-up
# - forward
# - cross-entropy
# - backward
# - optimizer.step
# - time and VRAM measurement

import gc
import time
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# 1. MODEL AND GPU CHECK
# ============================================================

assert "model" in globals(), (
    "First run the cell that creates the ANKA-50M model."
)

assert torch.cuda.is_available(), (
    "CUDA GPU not found."
)

# Device and config are taken directly from the model.
device = next(model.parameters()).device
cfg = model.cfg

assert device.type == "cuda", (
    f"Model is not on GPU: {device}"
)

gpu_name = torch.cuda.get_device_name(device)
total_vram_gb = (
    torch.cuda.get_device_properties(device).total_memory
    / 2**30
)

# For A100 40 GB and 80 GB, batch 64 is a safe starting point.
BATCH_SIZE = 64
SEQ_LEN = 512

WARMUP_STEPS = 5
MEASURE_STEPS = 50

NUM_WORKERS = 2
PREFETCH_FACTOR = 4

print("GPU             :", gpu_name)
print(f"Total VRAM     : {total_vram_gb:.2f} GB")
print("Model dtype     :", next(model.parameters()).dtype)
print("Batch / Sequence:", BATCH_SIZE, "/", SEQ_LEN)


# ============================================================
# 2. CLEAR OLD GPU MEMORY
# ============================================================

# Clear global tensor references remaining from previous failed cells.
for variable_name in [
    "optimizer",
    "gpu_batch",
    "cpu_batch",
    "input_ids",
    "logits",
    "loss",
    "loader",
    "loader_iterator",
]:
    globals().pop(variable_name, None)

model.zero_grad(set_to_none=True)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

memory_before_gb = (
    torch.cuda.memory_allocated(device)
    / 2**30
)

print(f"VRAM before test: {memory_before_gb:.2f} GB")


# ============================================================
# 3. CREATE OPTIMIZER IN THIS CELL
# ============================================================

model.train()

try:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4,
        betas=(0.9, 0.95),
        eps=1e-8,
        weight_decay=0.1,
        fused=True,
    )
    optimizer_name = "Fused AdamW"

except (TypeError, RuntimeError):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4,
        betas=(0.9, 0.95),
        eps=1e-8,
        weight_decay=0.1,
    )
    optimizer_name = "Standard AdamW"

print("Optimizer       :", optimizer_name)


# ============================================================
# 4. TRAINING STEP
# ============================================================

def run_training_step(token_batch):
    """
    Actual causal language-model training step:

    DataLoader
    -> GPU transfer
    -> forward
    -> cross-entropy
    -> backward
    -> optimizer.step
    """

    optimizer.zero_grad(set_to_none=True)

    # Embedding layer accepts int32 token indices.
    model_inputs = token_batch[:, :-1]

    # CrossEntropy targets must be int64/long.
    targets = token_batch[:, 1:].long()

    logits = model(model_inputs)

    loss = F.cross_entropy(
        logits.reshape(-1, cfg.vocab_size),
        targets.reshape(-1),
    )

    loss.backward()
    optimizer.step()

    loss_value = float(loss.detach())

    # Explicitly release the reference to the large logits tensor.
    del logits
    del loss

    return loss_value


# ============================================================
# 5. RANDOM TOKEN DATASET
# ============================================================

class RandomTokenDataset(Dataset):
    def __init__(
        self,
        number_of_sequences,
        sequence_length,
        vocabulary_size,
        seed=42,
    ):
        generator = torch.Generator()
        generator.manual_seed(seed)

        # CPU data is stored as int32.
        # RAM and transfer cost are halved compared to int64.
        self.tokens = torch.randint(
            low=0,
            high=vocabulary_size,
            size=(
                number_of_sequences,
                sequence_length,
            ),
            dtype=torch.int32,
            generator=generator,
        )

    def __len__(self):
        return self.tokens.shape[0]

    def __getitem__(self, index):
        return self.tokens[index]


number_of_sequences = (
    BATCH_SIZE
    * (WARMUP_STEPS + MEASURE_STEPS + 5)
)

dataset = RandomTokenDataset(
    number_of_sequences=number_of_sequences,
    sequence_length=SEQ_LEN,
    vocabulary_size=cfg.vocab_size,
    seed=42,
)

dataset_size_mb = (
    dataset.tokens.numel()
    * dataset.tokens.element_size()
    / 2**20
)

print(f"Number of sequences : {len(dataset):,}")
print(f"CPU dataset     : {dataset_size_mb:.2f} MB")


# ============================================================
# 6. DATALOADER
# ============================================================

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=(
        PREFETCH_FACTOR
        if NUM_WORKERS > 0
        else None
    ),
)

loader_iterator = iter(loader)

print("DataLoader batch:", len(loader))


# ============================================================
# 7. WARM-UP
# ============================================================

print("\nWarm-up starting...")

for warmup_step in range(WARMUP_STEPS):
    cpu_batch = next(loader_iterator)

    gpu_batch = cpu_batch.to(
        device=device,
        non_blocking=True,
    )

    warmup_loss = run_training_step(gpu_batch)

torch.cuda.synchronize()

print(f"Warm-up completed. Loss: {warmup_loss:.4f}")


# ============================================================
# 8. END-TO-END MEASUREMENT
# ============================================================

torch.cuda.reset_peak_memory_stats()

loader_wait_seconds = 0.0
transfer_submit_seconds = 0.0
measured_losses = []

torch.cuda.synchronize()
test_start_time = time.perf_counter()

for step in range(MEASURE_STEPS):

    # Time spent waiting for batch from DataLoader.
    loader_start_time = time.perf_counter()

    cpu_batch = next(loader_iterator)

    loader_wait_seconds += (
        time.perf_counter()
        - loader_start_time
    )

    # Send command to transfer CPU -> GPU.
    transfer_start_time = time.perf_counter()

    gpu_batch = cpu_batch.to(
        device=device,
        non_blocking=True,
    )

    transfer_submit_seconds += (
        time.perf_counter()
        - transfer_start_time
    )

    loss_value = run_training_step(gpu_batch)
    measured_losses.append(loss_value)

torch.cuda.synchronize()

total_elapsed_seconds = (
    time.perf_counter()
    - test_start_time
)


# ============================================================
# 9. CALCULATE RESULTS
# ============================================================

# SEQ_LEN-1 because the last token does not have a next-token target.
tokens_per_step = (
    BATCH_SIZE
    * (SEQ_LEN - 1)
)

total_processed_tokens = (
    tokens_per_step
    * MEASURE_STEPS
)

tokens_per_second = (
    total_processed_tokens
    / total_elapsed_seconds
)

average_step_ms = (
    total_elapsed_seconds
    * 1000
    / MEASURE_STEPS
)

average_loader_wait_ms = (
    loader_wait_seconds
    * 1000
    / MEASURE_STEPS
)

average_transfer_submit_ms = (
    transfer_submit_seconds
    * 1000
    / MEASURE_STEPS
)

peak_allocated_gb = (
    torch.cuda.max_memory_allocated(device)
    / 2**30
)

peak_reserved_gb = (
    torch.cuda.max_memory_reserved(device)
    / 2**30
)

average_loss = (
    sum(measured_losses)
    / len(measured_losses)
)


# ============================================================
# 10. RESULTS DISPLAY
# ============================================================

print("\n============================================")
print("ANKA-50M END-TO-END TRAINING THROUGHPUT")
print("============================================")
print("GPU                  :", gpu_name)
print(f"Total GPU VRAM       : {total_vram_gb:.2f} GB")
print("Model precision      :", next(model.parameters()).dtype)
print("Optimizer            :", optimizer_name)
print("Batch                :", BATCH_SIZE)
print("Sequence             :", SEQ_LEN)
print("Workers              :", NUM_WORKERS)
print("Pinned memory        : VAR")
print("Prefetch factor      :", PREFETCH_FACTOR)
print("Measurement step    :", MEASURE_STEPS)
print(f"Tokens/step          : {tokens_per_step:,}")
print(f"Total tokens         : {total_processed_tokens:,}")
print(f"Total time           : {total_elapsed_seconds:.2f} seconds")
print(f"Average step         : {average_step_ms:,.2f} ms")
print(f"Loader wait/step     : {average_loader_wait_ms:,.3f} ms")
print(f"Transfer submit/step : {average_transfer_submit_ms:,.3f} ms")
print(f"Training throughput  : {tokens_per_second:,.0f} token/s")
print(f"Peak allocated       : {peak_allocated_gb:.2f} GB")
print(f"Peak reserved        : {peak_reserved_gb:.2f} GB")
print(f"Average loss         : {average_loss:.4f}")
print("Forward              : VAR")
print("Backward             : VAR")
print("Optimizer.step       : VAR")
print("DataLoader           : VAR")


# ============================================================
# 11. COMPARE WITH PURE GPU RESULT
# ============================================================

previous_pure_gpu_speed = 142_207

pipeline_efficiency = (
    tokens_per_second
    / previous_pure_gpu_speed
    * 100
)

pipeline_loss_percent = max(
    0.0,
    (
        1
        - tokens_per_second
        / previous_pure_gpu_speed
    )
    * 100,
)

print("\n=== PIPELINE COMPARISON ===")
print(
    f"Pure GPU training    : "
    f"{previous_pure_gpu_speed:,.0f} token/s"
)
print(
    f"Including DataLoader : "
    f"{tokens_per_second:,.0f} token/s"
)
print(
    f"Pipeline efficiency : "
    f"%{pipeline_efficiency:.2f}"
)
print(
    f"Pipeline loss        : "
    f"%{pipeline_loss_percent:.2f}"
)